In [0]:
import importlib
import sys

# 1
if "../src" not in sys.path:
    sys.path.append("../src")

import bronze.nasa_firms_volume_ingestor as firms_mod

importlib.reload(firms_mod)

OUTPUT_JSON = "/Volumes/workspace/default/bronze_nasa_firms/data-2015-2026.json"

# 2
new_rows = firms_mod.ingest_firms_incremental(
    date_from="2026-11-20",
    date_to="2026-11-22",
    output_file=OUTPUT_JSON,
    source="VIIRS_NOAA20_NRT",
    splice_label="incremental_nrt",
    step_days=1,
    sleep_seconds=0.4,
)
print(f"Filas añadidas en este incremental: {new_rows}")

# 3
df_bronze = spark.read.json(OUTPUT_JSON)
print("Total filas en Bronze:", df_bronze.count())
df_bronze.selectExpr("min(acq_date) as fecha_min", "max(acq_date) as fecha_max").show()
df_bronze.groupBy("firms_source", "splice_segment").count().orderBy("firms_source", "splice_segment").show()

# --- Instrucción para regenerar los 11 años desde cero ---
# all_data = firms_mod.ingest_firms_spliced_2015_2026(
#     output_file=OUTPUT_JSON,
#     step_days=5,
#     sleep_seconds=0.5,
#     reset_file=True,
# )
# print(all_data)

In [ ]:
import os
import sys

SRC_PATH = os.path.abspath("../src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

sys.modules.pop("silver.firms_cleaner", None)
sys.modules.pop("silver", None)

import silver.firms_cleaner as firms_silver

catalog_name = "workspace"
schema_name = "default"
silver_table = f"{catalog_name}.{schema_name}.silver_nasa_firms"
bronze_path = "/Volumes/workspace/default/bronze_nasa_firms/data-2015-2026.json"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

df_bronze = spark.read.json(bronze_path)
df_silver = firms_silver.clean_firms_silver(df_bronze)
firms_silver.write_firms_silver_table(df_silver, table_name=silver_table)

df_check = spark.table(silver_table)
print("Tabla Silver en Unity Catalog:", silver_table)
print("Total filas:", df_check.count())
df_check.show(5, truncate=False)

In [ ]:
import os
import sys

SRC_PATH = os.path.abspath("../src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

sys.modules.pop("gold.firms_analytics", None)
sys.modules.pop("gold", None)

import gold.firms_analytics as firms_gold

df_silver = spark.table("workspace.default.silver_nasa_firms")

df_monthly = firms_gold.build_firms_monthly(df_silver)
df_grid = firms_gold.build_firms_grid(df_silver, step=0.05)
df_prev_7d = firms_gold.build_firms_prev_7d(df_silver, step=0.05)

df_monthly.show(12, truncate=False)
df_grid.show(10, truncate=False)
df_prev_7d.filter("prev_7d_detections > 0").orderBy("day_frp_sum", ascending=False).show(10, truncate=False)

In [ ]:
gold_monthly_table = "workspace.default.gold_firms_monthly"
gold_grid_table = "workspace.default.gold_firms_grid"
gold_prev_7d_table = "workspace.default.gold_firms_prev_7d"

df_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_monthly_table)
df_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_grid_table)
df_prev_7d.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_prev_7d_table)

print("Tablas Gold guardadas en Unity Catalog.")

In [ ]:
from pyspark.sql import functions as F

df_prev_7d_saved = spark.table("workspace.default.gold_firms_prev_7d")

# Ejemplo incendio de Burgohondo
target_lat = 40.35
target_lon = -4.75

(
    df_prev_7d_saved
    .filter((F.col("lat_cell") == target_lat) & (F.col("lon_cell") == target_lon))
    .orderBy("acq_date")
    .show(50, truncate=False)
)